# Financial Analyzer Agent - Infrastructure Setup

This notebook guides you through setting up all AWS infrastructure needed for the Financial Analyzer Agent.

## Architecture Overview

We'll create:
1. **DynamoDB Table** - Project budget data storage
2. **S3 Bucket** - Quarterly financial data (XLSX)
3. **Lambda Function** - Query interface for DynamoDB
4. **Cognito User Pool** - JWT authentication for AgentCore Gateway
5. **AgentCore Gateway** - Converts Lambda to MCP tools with JWT authorizer

## Prerequisites

- AWS CLI configured (`aws configure`)
- Python 3.9+ with boto3 installed
- Appropriate AWS permissions
- Bedrock Model Access enabled in your region

## Configuration

Set your AWS region and resource prefix:

In [4]:
import boto3
from boto3.session import Session
import json
import time
import zipfile
import io
from datetime import datetime

# Configuration
AWS_REGION = "us-east-1"  # Change if needed
RESOURCE_PREFIX = "finance-analyzer"
AWS_PROFILE = "eliorf-Admin"  # Change to your AWS SSO profile name

# Create boto3 session with profile
boto_session = Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)

# Get account ID
sts = boto_session.client('sts')
account_id = sts.get_caller_identity()['Account']

# Derived names
DYNAMODB_TABLE = f"{RESOURCE_PREFIX}-projects-budget"
S3_BUCKET = f"{RESOURCE_PREFIX}-data-{account_id}"
LAMBDA_FUNCTION = f"{RESOURCE_PREFIX}-project-queries"
LAMBDA_ROLE = f"{RESOURCE_PREFIX}-lambda-role"
COGNITO_USER_POOL = f"{RESOURCE_PREFIX}-user-pool"
COGNITO_DOMAIN = f"{RESOURCE_PREFIX}-{int(time.time())}"
GATEWAY_NAME = f"{RESOURCE_PREFIX}-gateway"
IDENTITY_NAME = f"{RESOURCE_PREFIX}-identity"

print(f"✓ Region: {AWS_REGION}")
print(f"✓ Profile: {AWS_PROFILE}")
print(f"✓ Account: {account_id}")
print(f"✓ Prefix: {RESOURCE_PREFIX}")
print(f"✓ DynamoDB Table: {DYNAMODB_TABLE}")
print(f"✓ S3 Bucket: {S3_BUCKET}")
print(f"✓ Lambda Function: {LAMBDA_FUNCTION}")

✓ Region: us-east-1
✓ Profile: eliorf-Admin
✓ Account: 961341522526
✓ Prefix: finance-analyzer
✓ DynamoDB Table: finance-analyzer-projects-budget
✓ S3 Bucket: finance-analyzer-data-961341522526
✓ Lambda Function: finance-analyzer-project-queries


## Step 1: Create DynamoDB Table

Create the DynamoDB table with project budget schema:

In [5]:
# Initialize DynamoDB client
dynamodb = boto_session.client('dynamodb', region_name=AWS_REGION)

# Create table
try:
    response = dynamodb.create_table(
        TableName=DYNAMODB_TABLE,
        KeySchema=[
            {'AttributeName': 'projectId', 'KeyType': 'HASH'}  # Partition key
        ],
        AttributeDefinitions=[
            {'AttributeName': 'projectId', 'AttributeType': 'S'}
        ],
        BillingMode='PAY_PER_REQUEST'  # On-demand pricing
    )
    
    print(f"✓ Creating DynamoDB table: {DYNAMODB_TABLE}")
    
    # Wait for table to be active
    waiter = dynamodb.get_waiter('table_exists')
    waiter.wait(TableName=DYNAMODB_TABLE)
    
    print(f"✓ Table created successfully")
    print(f"  ARN: {response['TableDescription']['TableArn']}")
    
except dynamodb.exceptions.ResourceInUseException:
    print(f"⚠ Table {DYNAMODB_TABLE} already exists")
except Exception as e:
    print(f"✗ Error creating table: {e}")
    raise

✓ Creating DynamoDB table: finance-analyzer-projects-budget
✓ Table created successfully
  ARN: arn:aws:dynamodb:us-east-1:961341522526:table/finance-analyzer-projects-budget


## Step 2: Load Sample Data into DynamoDB

Load 10 sample projects from the data file:

In [6]:
# Load sample data
with open('data/project-budget.json', 'r') as f:
    data = json.load(f)

# Batch write items
table_name = DYNAMODB_TABLE
items = data[list(data.keys())[0]]  # Get items from first key

try:
    # DynamoDB batch_write_item in chunks of 25
    for i in range(0, len(items), 25):
        batch = items[i:i+25]
        dynamodb.batch_write_item(
            RequestItems={
                DYNAMODB_TABLE: batch
            }
        )
    
    print(f"✓ Loaded {len(items)} projects into DynamoDB")
    
    # Verify by scanning
    response = dynamodb.scan(TableName=DYNAMODB_TABLE, Select='COUNT')
    print(f"✓ Verified: {response['Count']} items in table")
    
except Exception as e:
    print(f"✗ Error loading data: {e}")
    raise

✓ Loaded 10 projects into DynamoDB
✓ Verified: 10 items in table


## Step 3: Create S3 Bucket

Create an S3 bucket for quarterly financial data:

In [7]:
# Initialize S3 client
s3 = boto_session.client('s3', region_name=AWS_REGION)

try:
    # Create bucket
    if AWS_REGION == 'us-east-1':
        s3.create_bucket(Bucket=S3_BUCKET)
    else:
        s3.create_bucket(
            Bucket=S3_BUCKET,
            CreateBucketConfiguration={'LocationConstraint': AWS_REGION}
        )
    
    print(f"✓ Created S3 bucket: {S3_BUCKET}")
    
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"⚠ Bucket {S3_BUCKET} already exists and is owned by you")
except s3.exceptions.BucketAlreadyExists:
    print(f"✗ Bucket name {S3_BUCKET} is already taken by another account")
    raise
except Exception as e:
    print(f"✗ Error creating bucket: {e}")
    raise

✓ Created S3 bucket: finance-analyzer-data-961341522526


## Step 4: Upload Quarterly Data to S3

Upload the XLSX file with financial data:

In [8]:
# Upload XLSX file
xlsx_file = 'data/quarterly_results.xlsx'
s3_key = 'quarterly-data/quarterly_results.xlsx'

try:
    s3.upload_file(xlsx_file, S3_BUCKET, s3_key)
    print(f"✓ Uploaded {xlsx_file} to s3://{S3_BUCKET}/{s3_key}")
    
    # Verify upload
    response = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)
    print(f"✓ File size: {response['ContentLength']:,} bytes")
    
except Exception as e:
    print(f"✗ Error uploading file: {e}")
    raise

✓ Uploaded data/quarterly_results.xlsx to s3://finance-analyzer-data-961341522526/quarterly-data/quarterly_results.xlsx
✓ File size: 13,599 bytes


## Step 5: Create IAM Role for Lambda

Create an execution role with permissions for DynamoDB access:

In [9]:
# Initialize IAM client
iam = boto_session.client('iam', region_name=AWS_REGION)

# Trust policy for Lambda
assume_role_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}

try:
    # Create role
    role_response = iam.create_role(
        RoleName=LAMBDA_ROLE,
        AssumeRolePolicyDocument=json.dumps(assume_role_policy),
        Description='Execution role for finance analyzer Lambda function'
    )
    
    role_arn = role_response['Role']['Arn']
    print(f"✓ Created IAM role: {LAMBDA_ROLE}")
    print(f"  ARN: {role_arn}")
    
except iam.exceptions.EntityAlreadyExistsException:
    print(f"⚠ Role {LAMBDA_ROLE} already exists")
    role_arn = iam.get_role(RoleName=LAMBDA_ROLE)['Role']['Arn']
except Exception as e:
    print(f"✗ Error creating role: {e}")
    raise

# Attach basic Lambda execution policy
try:
    iam.attach_role_policy(
        RoleName=LAMBDA_ROLE,
        PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
    )
    print(f"✓ Attached AWSLambdaBasicExecutionRole policy")
except Exception as e:
    print(f"⚠ Policy may already be attached: {e}")

✓ Created IAM role: finance-analyzer-lambda-role
  ARN: arn:aws:iam::961341522526:role/finance-analyzer-lambda-role
✓ Attached AWSLambdaBasicExecutionRole policy


## Step 6: Add DynamoDB Permissions to Lambda Role

Create and attach an inline policy for DynamoDB access:

In [10]:
# Get table ARN
table_description = dynamodb.describe_table(TableName=DYNAMODB_TABLE)
table_arn = table_description['Table']['TableArn']

# DynamoDB access policy
dynamodb_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "dynamodb:GetItem",
                "dynamodb:Scan",
                "dynamodb:Query"
            ],
            "Resource": table_arn
        }
    ]
}

try:
    iam.put_role_policy(
        RoleName=LAMBDA_ROLE,
        PolicyName='DynamoDBAccess',
        PolicyDocument=json.dumps(dynamodb_policy)
    )
    print(f"✓ Added DynamoDB access policy to role")
    print(f"  Table: {DYNAMODB_TABLE}")
    
except Exception as e:
    print(f"✗ Error adding policy: {e}")
    raise

# Wait for role to propagate
print("⏳ Waiting 10 seconds for IAM role to propagate...")
time.sleep(10)
print("✓ Role ready")

✓ Added DynamoDB access policy to role
  Table: finance-analyzer-projects-budget
⏳ Waiting 10 seconds for IAM role to propagate...
✓ Role ready


## Step 7: Create Lambda Function

Package and deploy the Lambda function code:

In [12]:
# Read Lambda code
with open('../src/project-budget-lambda.py', 'r') as f:
    lambda_code = f.read()

# Update table name in Lambda code
lambda_code = lambda_code.replace(
    "table = dynamodb.Table('agentcore-demo-projects-budget')",
    f"table = dynamodb.Table('{DYNAMODB_TABLE}')"
)

# Create deployment package
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zip_file:
    zip_file.writestr('lambda_function.py', lambda_code)

zip_buffer.seek(0)
deployment_package = zip_buffer.read()

print(f"✓ Created deployment package ({len(deployment_package):,} bytes)")

✓ Created deployment package (1,374 bytes)


In [13]:
# Initialize Lambda client
lambda_client = boto_session.client('lambda', region_name=AWS_REGION)

try:
    # Create Lambda function
    response = lambda_client.create_function(
        FunctionName=LAMBDA_FUNCTION,
        Runtime='python3.11',
        Role=role_arn,
        Handler='lambda_function.lambda_handler',
        Code={'ZipFile': deployment_package},
        Description='Query interface for project budget DynamoDB table',
        Timeout=30,
        MemorySize=256,
        Environment={
            'Variables': {
                'DYNAMODB_TABLE': DYNAMODB_TABLE
            }
        }
    )
    
    lambda_arn = response['FunctionArn']
    print(f"✓ Created Lambda function: {LAMBDA_FUNCTION}")
    print(f"  ARN: {lambda_arn}")
    
except lambda_client.exceptions.ResourceConflictException:
    print(f"⚠ Lambda function {LAMBDA_FUNCTION} already exists")
    # Update existing function
    lambda_client.update_function_code(
        FunctionName=LAMBDA_FUNCTION,
        ZipFile=deployment_package
    )
    print(f"✓ Updated existing Lambda function code")
    
    response = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION)
    lambda_arn = response['Configuration']['FunctionArn']
    
except Exception as e:
    print(f"✗ Error creating Lambda function: {e}")
    raise

✓ Created Lambda function: finance-analyzer-project-queries
  ARN: arn:aws:lambda:us-east-1:961341522526:function:finance-analyzer-project-queries


## Step 8: Create Cognito User Pool

Create a Cognito User Pool for AgentCore Identity authentication:

In [14]:
# Initialize Cognito client
cognito = boto_session.client('cognito-idp', region_name=AWS_REGION)

# Check if user pool already exists
pools = cognito.list_user_pools(MaxResults=60)['UserPools']
existing_pool = next((p for p in pools if p['Name'] == COGNITO_USER_POOL), None)

if existing_pool:
    user_pool_id = existing_pool['Id']
    print(f"⚠ Cognito User Pool '{COGNITO_USER_POOL}' already exists")
    print(f"  Pool ID: {user_pool_id}")
else:
    try:
        # Create User Pool
        response = cognito.create_user_pool(
            PoolName=COGNITO_USER_POOL,
            Policies={
                'PasswordPolicy': {
                    'MinimumLength': 8,
                    'RequireUppercase': False,
                    'RequireLowercase': False,
                    'RequireNumbers': False,
                    'RequireSymbols': False
                }
            },
            AutoVerifiedAttributes=[],
            UserPoolTags={
                'Project': 'finance-analyzer',
                'Purpose': 'AgentCore Identity'
            }
        )
        
        user_pool_id = response['UserPool']['Id']
        print(f"✓ Created Cognito User Pool: {COGNITO_USER_POOL}")
        print(f"  Pool ID: {user_pool_id}")
        
    except Exception as e:
        print(f"✗ Error creating user pool: {e}")
        raise

✓ Created Cognito User Pool: finance-analyzer-user-pool
  Pool ID: us-east-1_iyiwVBxG5


## Step 9: Create Cognito Domain

Create a domain for OAuth endpoints:

In [15]:
# Check if user pool already has a domain
try:
    pool_details = cognito.describe_user_pool(UserPoolId=user_pool_id)
    existing_domain = pool_details['UserPool'].get('Domain')

    if existing_domain:
        # Use existing domain
        COGNITO_DOMAIN = existing_domain
        print(f"⚠ User Pool already has domain: {COGNITO_DOMAIN}")
        print(f"  OAuth URL: https://{COGNITO_DOMAIN}.auth.{AWS_REGION}.amazoncognito.com")
    else:
        # Create new domain with static name (no timestamp)
        COGNITO_DOMAIN = f"{RESOURCE_PREFIX}-domain"

        try:
            cognito.create_user_pool_domain(
                Domain=COGNITO_DOMAIN,
                UserPoolId=user_pool_id
            )
            print(f"✓ Created Cognito domain: {COGNITO_DOMAIN}")
            print(f"  OAuth URL: https://{COGNITO_DOMAIN}.auth.{AWS_REGION}.amazoncognito.com")

        except cognito.exceptions.InvalidParameterException as e:
            if 'already exists' in str(e).lower():
                # Domain name taken, try with timestamp
                COGNITO_DOMAIN = f"{RESOURCE_PREFIX}-{int(time.time())}"
                cognito.create_user_pool_domain(
                    Domain=COGNITO_DOMAIN,
                    UserPoolId=user_pool_id
                )
                print(f"✓ Created Cognito domain: {COGNITO_DOMAIN}")
                print(f"  OAuth URL: https://{COGNITO_DOMAIN}.auth.{AWS_REGION}.amazoncognito.com")
            else:
                raise

except Exception as e:
    print(f"✗ Error with Cognito domain: {e}")
    raise

✓ Created Cognito domain: finance-analyzer-domain
  OAuth URL: https://finance-analyzer-domain.auth.us-east-1.amazoncognito.com


In [16]:
try:
    # Create Resource Server first (needed for scopes)
    resource_server_id = f"{RESOURCE_PREFIX}-resource-server"
    try:
        cognito.create_resource_server(
            UserPoolId=user_pool_id,
            Identifier=resource_server_id,
            Name=f"{RESOURCE_PREFIX}-api",
            Scopes=[
                {
                    'ScopeName': 'read',
                    'ScopeDescription': 'Read access'
                }
            ]
        )
        print(f"✓ Created resource server: {resource_server_id}")
    except cognito.exceptions.InvalidParameterException as e:
        if 'already exists' in str(e).lower():
            print(f"⚠ Resource server {resource_server_id} already exists")
        else:
            raise
    
    # Create App Client with scopes
    response = cognito.create_user_pool_client(
        UserPoolId=user_pool_id,
        ClientName=f"{RESOURCE_PREFIX}-app-client",
        GenerateSecret=True,
        ExplicitAuthFlows=['ALLOW_REFRESH_TOKEN_AUTH'],
        AllowedOAuthFlows=['client_credentials'],
        AllowedOAuthFlowsUserPoolClient=True,
        AllowedOAuthScopes=[f"{resource_server_id}/read"],
        PreventUserExistenceErrors='ENABLED'
    )
    
    client_id = response['UserPoolClient']['ClientId']
    client_secret = response['UserPoolClient']['ClientSecret']
    
    print(f"✓ Created App Client")
    print(f"  Client ID: {client_id}")
    print(f"  Client Secret: {client_secret[:20]}...")
    
except Exception as e:
    print(f"✗ Error creating app client: {e}")
    raise

✓ Created resource server: finance-analyzer-resource-server
✓ Created App Client
  Client ID: 4i0g0cncrlountm5452equoen7
  Client Secret: viqdb11r80d1aib9pbt4...


## Step 11: Create IAM Role for Gateway

Create an IAM role that AgentCore Gateway will assume:

In [17]:
# Create IAM role for Gateway
GATEWAY_ROLE = f"{RESOURCE_PREFIX}-gateway-role"

# Trust policy for Bedrock services (CRITICAL: Must include both services)
gateway_assume_role_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "bedrock.amazonaws.com",
                    "bedrock-agentcore.amazonaws.com"  # Required for AgentCore Gateway
                ]
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

try:
    # Create role
    role_response = iam.create_role(
        RoleName=GATEWAY_ROLE,
        AssumeRolePolicyDocument=json.dumps(gateway_assume_role_policy),
        Description='Execution role for AgentCore Gateway'
    )
    
    gateway_role_arn = role_response['Role']['Arn']
    print(f"✓ Created IAM role: {GATEWAY_ROLE}")
    print(f"  ARN: {gateway_role_arn}")
    
except iam.exceptions.EntityAlreadyExistsException:
    print(f"⚠ Role {GATEWAY_ROLE} already exists")
    gateway_role_arn = iam.get_role(RoleName=GATEWAY_ROLE)['Role']['Arn']
    
    # Update trust policy to ensure it has both services
    try:
        iam.update_assume_role_policy(
            RoleName=GATEWAY_ROLE,
            PolicyDocument=json.dumps(gateway_assume_role_policy)
        )
        print(f"  ✓ Updated trust policy to include bedrock-agentcore.amazonaws.com")
    except Exception as e:
        print(f"  ⚠ Trust policy update: {str(e)}")
        
except Exception as e:
    print(f"✗ Error creating role: {e}")
    raise

# Add Lambda invocation permission
gateway_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "lambda:InvokeFunction",
            "Resource": lambda_arn
        }
    ]
}

try:
    iam.put_role_policy(
        RoleName=GATEWAY_ROLE,
        PolicyName='LambdaInvokeAccess',
        PolicyDocument=json.dumps(gateway_policy)
    )
    print(f"✓ Added Lambda invoke policy to Gateway role")
except Exception as e:
    print(f"⚠ Policy may already be attached: {str(e)}")

# Wait for role to propagate
print("⏳ Waiting 10 seconds for IAM role to propagate...")
time.sleep(10)
print("✓ Role ready")

✓ Created IAM role: finance-analyzer-gateway-role
  ARN: arn:aws:iam::961341522526:role/finance-analyzer-gateway-role
✓ Added Lambda invoke policy to Gateway role
⏳ Waiting 10 seconds for IAM role to propagate...
✓ Role ready


## Step 12: Create AgentCore Gateway with JWT Authorizer

Create AgentCore Gateway with Cognito JWT authentication:

In [18]:
# Initialize Bedrock AgentCore Control client
bedrock_agentcore = boto_session.client('bedrock-agentcore-control', region_name=AWS_REGION)

# Get Cognito OIDC discovery URL
discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

try:
    # Create Gateway with JWT authorizer
    # Note: Cognito client credentials flow doesn't include 'aud' claim, only 'client_id'
    # So we only specify allowedClients, not allowedAudience
    response = bedrock_agentcore.create_gateway(
        name=GATEWAY_NAME,
        roleArn=gateway_role_arn,
        protocolType='MCP',  # Must be MCP for AgentCore Gateway
        authorizerType='CUSTOM_JWT',
        authorizerConfiguration={
            'customJWTAuthorizer': {
                'discoveryUrl': discovery_url,
                'allowedClients': [client_id]
                # Note: Not using allowedAudience since Cognito client credentials JWT
                # only has 'client_id' claim, not 'aud' claim
            }
        },
        description='Gateway for project budget queries'
    )
    
    gateway_id = response['gatewayId']
    gateway_arn = response['gatewayArn']
    gateway_url = response['gatewayUrl']
    
    print(f"✓ Created AgentCore Gateway: {GATEWAY_NAME}")
    print(f"  Gateway ID: {gateway_id}")
    print(f"  Gateway ARN: {gateway_arn}")
    print(f"  Gateway URL: {gateway_url}")
    print(f"  Authorizer: JWT (Cognito - client_id validation)")
    
except Exception as e:
    if 'ConflictException' in str(type(e).__name__):
        print(f"⚠ Gateway {GATEWAY_NAME} already exists")
        # List and find existing gateway
        gateways = bedrock_agentcore.list_gateways().get('items', [])
        gateway = next((g for g in gateways if g['name'] == GATEWAY_NAME), None)
        if gateway:
            gateway_id = gateway['gatewayId']
            # Get full details using gateway ID
            details = bedrock_agentcore.get_gateway(gatewayIdentifier=gateway_id)
            gateway_arn = details['gatewayArn']
            gateway_url = details['gatewayUrl']
            
            # Update authorizer config to remove allowedAudience if it exists
            try:
                bedrock_agentcore.update_gateway(
                    gatewayIdentifier=gateway_id,
                    name=details['name'],
                    roleArn=details['roleArn'],
                    protocolType=details['protocolType'],
                    authorizerType=details['authorizerType'],
                    authorizerConfiguration={
                        'customJWTAuthorizer': {
                            'discoveryUrl': discovery_url,
                            'allowedClients': [client_id]
                        }
                    }
                )
                print(f"  ✓ Updated Gateway authorizer config (removed allowedAudience)")
            except Exception as update_error:
                print(f"  ⚠ Could not update authorizer: {str(update_error)}")
            
            print(f"  Using existing gateway: {gateway_id}")
            print(f"  Gateway URL: {gateway_url}")
        else:
            print(f"✗ Could not find existing gateway")
            raise Exception("Could not find existing gateway")
    else:
        print(f"✗ Error creating gateway: {e}")
        raise

✓ Created AgentCore Gateway: finance-analyzer-gateway
  Gateway ID: finance-analyzer-gateway-mbufomquuo
  Gateway ARN: arn:aws:bedrock-agentcore:us-east-1:961341522526:gateway/finance-analyzer-gateway-mbufomquuo
  Gateway URL: https://finance-analyzer-gateway-mbufomquuo.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
  Authorizer: JWT (Cognito - client_id validation)


## Step 13: Create Gateway Target for Lambda

Connect the Lambda function to the Gateway as a target:

In [19]:
# Load tool definitions
with open('schemas/gateway-projects-budget.json', 'r') as f:
    tool_definitions = json.load(f)

TARGET_NAME = f"{RESOURCE_PREFIX}-lambda-target"

try:
    # Create Gateway Target for Lambda
    target_response = bedrock_agentcore.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name=TARGET_NAME,
        targetConfiguration={
            'mcp': {
                'lambda': {
                    'lambdaArn': lambda_arn,
                    'toolSchema': {
                        'inlinePayload': tool_definitions
                    }
                }
            }
        },
        credentialProviderConfigurations=[
            {
                'credentialProviderType': 'GATEWAY_IAM_ROLE'
            }
        ],
        description='Lambda target for project budget queries'
    )
    
    target_id = target_response['targetId']
    print(f"✓ Created Gateway Target: {TARGET_NAME}")
    print(f"  Target ID: {target_id}")
    print(f"  Lambda ARN: {lambda_arn}")
    print(f"  Tools: {len(tool_definitions)}")
    
except Exception as e:
    if 'ConflictException' in str(type(e).__name__):
        print(f"⚠ Gateway target {TARGET_NAME} already exists")
        # List and find existing target
        targets = bedrock_agentcore.list_gateway_targets(gatewayIdentifier=gateway_id)
        target_list = targets.get('items', [])
        target = next((t for t in target_list if t['name'] == TARGET_NAME), None)
        if target:
            target_id = target['targetId']
            print(f"  Using existing target: {target_id}")
        elif target_list:
            # Use first target if name doesn't match
            target_id = target_list[0]['targetId']
            print(f"  Using existing target: {target_id}")
        else:
            print(f"✗ Could not find existing target")
            raise Exception("Could not find existing target")
    else:
        print(f"✗ Error creating gateway target: {e}")
        raise

✓ Created Gateway Target: finance-analyzer-lambda-target
  Target ID: BL3YSVNHLZ
  Lambda ARN: arn:aws:lambda:us-east-1:961341522526:function:finance-analyzer-project-queries
  Tools: 5


## Step 14: Store Configuration in SSM Parameter Store

Save all configuration parameters for agent usage:

In [20]:
# Initialize SSM client
ssm = boto_session.client('ssm', region_name=AWS_REGION)

# Configuration parameters
parameters = {
    'dynamodb_table': DYNAMODB_TABLE,
    's3_bucket': S3_BUCKET,
    's3_quarterly_data_path': f's3://{S3_BUCKET}/quarterly-data/',
    'lambda_function': lambda_arn,
    'user_pool_id': user_pool_id,
    'client_id': client_id,
    'client_secret': client_secret,
    'cognito_domain': COGNITO_DOMAIN,
    'gateway_id': gateway_id,
    'gateway_arn': gateway_arn,
    'gateway_url': gateway_url
}

# Store each parameter
for key, value in parameters.items():
    param_name = f"/{RESOURCE_PREFIX}/{key}"
    param_type = 'SecureString' if 'secret' in key else 'String'
    
    try:
        ssm.put_parameter(
            Name=param_name,
            Value=value,
            Type=param_type,
            Overwrite=True,
            Description=f'Finance Analyzer - {key}'
        )
        print(f"✓ Stored: {param_name}")
    except Exception as e:
        print(f"✗ Error storing {param_name}: {e}")

print(f"\n✓ Configuration stored in SSM Parameter Store")
print(f"  Prefix: /{RESOURCE_PREFIX}/")

✓ Stored: /finance-analyzer/dynamodb_table
✓ Stored: /finance-analyzer/s3_bucket
✓ Stored: /finance-analyzer/s3_quarterly_data_path
✓ Stored: /finance-analyzer/lambda_function
✓ Stored: /finance-analyzer/user_pool_id
✓ Stored: /finance-analyzer/client_id
✓ Stored: /finance-analyzer/client_secret
✓ Stored: /finance-analyzer/cognito_domain
✓ Stored: /finance-analyzer/gateway_id
✓ Stored: /finance-analyzer/gateway_arn
✓ Stored: /finance-analyzer/gateway_url

✓ Configuration stored in SSM Parameter Store
  Prefix: /finance-analyzer/


## Step 15: Verify Setup

Summary of all created resources:

In [21]:
print("\n" + "="*70)
print("INFRASTRUCTURE SETUP COMPLETE")
print("="*70)

print(f"\n✓ DynamoDB Table: {DYNAMODB_TABLE}")
response = dynamodb.scan(TableName=DYNAMODB_TABLE, Select='COUNT')
print(f"  - Items: {response['Count']} projects")

print(f"\n✓ S3 Bucket: {S3_BUCKET}")
print(f"  - File: s3://{S3_BUCKET}/{s3_key}")

print(f"\n✓ Lambda Function: {LAMBDA_FUNCTION}")
print(f"  - ARN: {lambda_arn}")
print(f"  - Role: {LAMBDA_ROLE}")

print(f"\n✓ Cognito User Pool: {user_pool_id}")
print(f"  - Domain: {COGNITO_DOMAIN}")
print(f"  - Client ID: {client_id}")
print(f"  - Discovery URL: {discovery_url}")

print(f"\n✓ AgentCore Gateway: {gateway_id}")
print(f"  - ARN: {gateway_arn}")
print(f"  - URL: {gateway_url}")
print(f"  - Authorizer: JWT (Cognito)")
print(f"  - Role: {GATEWAY_ROLE}")
print(f"  - Tools: {len(tool_definitions)}")

print(f"\n✓ SSM Parameters: /{RESOURCE_PREFIX}/*")

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("\n1. Install Python dependencies:")
print("   pip install -r requirements.txt")
print("\n2. Run the agent:")
print("   python demo1/analyst_assistant_unified_strands.py")
print("\n3. Try example queries:")
print('   - "Show me project PROJ-001"')
print('   - "List all Marketing projects"')
print('   - "What was Q4 2023 revenue?"')
print("\n" + "="*70)


INFRASTRUCTURE SETUP COMPLETE

✓ DynamoDB Table: finance-analyzer-projects-budget
  - Items: 10 projects

✓ S3 Bucket: finance-analyzer-data-961341522526
  - File: s3://finance-analyzer-data-961341522526/quarterly-data/quarterly_results.xlsx

✓ Lambda Function: finance-analyzer-project-queries
  - ARN: arn:aws:lambda:us-east-1:961341522526:function:finance-analyzer-project-queries
  - Role: finance-analyzer-lambda-role

✓ Cognito User Pool: us-east-1_iyiwVBxG5
  - Domain: finance-analyzer-domain
  - Client ID: 4i0g0cncrlountm5452equoen7
  - Discovery URL: https://cognito-idp.us-east-1.amazonaws.com/us-east-1_iyiwVBxG5/.well-known/openid-configuration

✓ AgentCore Gateway: finance-analyzer-gateway-mbufomquuo
  - ARN: arn:aws:bedrock-agentcore:us-east-1:961341522526:gateway/finance-analyzer-gateway-mbufomquuo
  - URL: https://finance-analyzer-gateway-mbufomquuo.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
  - Authorizer: JWT (Cognito)
  - Role: finance-analyzer-gateway-role
  -

---

# Cleanup - Delete All Resources

⚠️ **WARNING**: This cell will delete **all** created resources from your AWS account.

Run this cell **only** when you want to completely tear down the infrastructure.

In [36]:
print("\n" + "="*70)
print("STARTING CLEANUP - DELETING ALL RESOURCES")
print("="*70 + "\n")

cleanup_errors = []

# 1. Delete AgentCore Gateway (delete targets first, then gateway)
print("[1/7] Deleting AgentCore Gateway...")
try:
    # Delete gateway targets first
    try:
        targets_response = bedrock_agentcore.list_gateway_targets(gatewayIdentifier=gateway_id)
        target_list = targets_response.get('items', [])
        
        if target_list:
            for target in target_list:
                target_id = target['targetId']
                try:
                    bedrock_agentcore.delete_gateway_target(
                        gatewayIdentifier=gateway_id,
                        targetId=target_id
                    )
                    print(f"  ✓ Deleted gateway target: {target_id}")
                except Exception as e:
                    if 'NotFound' in str(type(e).__name__) or 'ResourceNotFound' in str(e):
                        print(f"  ℹ Target {target_id} already deleted")
                    else:
                        raise
            
            # Wait for targets to be fully deleted
            print(f"  ⏳ Waiting 10 seconds for target deletion to complete...")
            time.sleep(10)
            
            # Verify targets are deleted
            remaining = bedrock_agentcore.list_gateway_targets(gatewayIdentifier=gateway_id)
            if remaining.get('items'):
                print(f"  ⚠ Warning: {len(remaining['items'])} targets still exist")
            else:
                print(f"  ✓ All targets deleted")
        else:
            print(f"  ℹ No gateway targets to delete")
    except Exception as e:
        if 'NotFound' in str(type(e).__name__) or 'ResourceNotFound' in str(e):
            print(f"  ℹ Gateway targets already deleted or gateway not found")
        else:
            raise
    
    # Delete the gateway itself
    bedrock_agentcore.delete_gateway(gatewayIdentifier=gateway_id)
    print(f"  ✓ Deleted Gateway: {gateway_id}")
    
except Exception as e:
    if 'NotFound' in str(type(e).__name__) or 'ResourceNotFound' in str(e):
        print(f"  ℹ Gateway {gateway_id} already deleted")
    else:
        error_msg = f"Gateway deletion: {str(e)}"
        cleanup_errors.append(error_msg)
        print(f"  ⚠ {error_msg}")

# 2. Delete Gateway IAM Role
print("\n[2/7] Deleting Gateway IAM role...")

try:
    iam.delete_role_policy(
        RoleName=GATEWAY_ROLE,
        PolicyName='LambdaInvokeAccess'
    )
    print(f"  ✓ Deleted inline policy: LambdaInvokeAccess")
except iam.exceptions.NoSuchEntityException:
    print(f"  ℹ Policy LambdaInvokeAccess already deleted")
except Exception as e:
    print(f"  ⚠ Inline policy deletion: {str(e)}")

try:
    iam.delete_role(RoleName=GATEWAY_ROLE)
    print(f"  ✓ Deleted role: {GATEWAY_ROLE}")
except iam.exceptions.NoSuchEntityException:
    print(f"  ℹ Role {GATEWAY_ROLE} already deleted")
except Exception as e:
    error_msg = f"Gateway role deletion: {str(e)}"
    cleanup_errors.append(error_msg)
    print(f"  ⚠ {error_msg}")

# 3. Delete Cognito Resources
print("\n[3/7] Deleting Cognito resources...")

# Delete app client
try:
    cognito.delete_user_pool_client(
        UserPoolId=user_pool_id,
        ClientId=client_id
    )
    print(f"  ✓ Deleted app client: {client_id}")
except cognito.exceptions.ResourceNotFoundException:
    print(f"  ℹ App client {client_id} already deleted")
except Exception as e:
    print(f"  ⚠ App client deletion: {str(e)}")

# Delete resource server
try:
    resource_server_id = f"{RESOURCE_PREFIX}-resource-server"
    cognito.delete_resource_server(
        UserPoolId=user_pool_id,
        Identifier=resource_server_id
    )
    print(f"  ✓ Deleted resource server: {resource_server_id}")
except cognito.exceptions.ResourceNotFoundException:
    print(f"  ℹ Resource server {resource_server_id} already deleted")
except Exception as e:
    print(f"  ⚠ Resource server deletion: {str(e)}")

# Delete domain
try:
    cognito.delete_user_pool_domain(
        Domain=COGNITO_DOMAIN,
        UserPoolId=user_pool_id
    )
    print(f"  ✓ Deleted domain: {COGNITO_DOMAIN}")
except (cognito.exceptions.ResourceNotFoundException, cognito.exceptions.InvalidParameterException) as e:
    if 'No such domain' in str(e) or 'does not exist' in str(e):
        print(f"  ℹ Domain {COGNITO_DOMAIN} already deleted")
    else:
        print(f"  ⚠ Domain deletion: {str(e)}")
except Exception as e:
    print(f"  ⚠ Domain deletion: {str(e)}")

# Delete user pool
try:
    cognito.delete_user_pool(UserPoolId=user_pool_id)
    print(f"  ✓ Deleted user pool: {user_pool_id}")
except cognito.exceptions.ResourceNotFoundException:
    print(f"  ℹ User pool {user_pool_id} already deleted")
except Exception as e:
    error_msg = f"User pool deletion: {str(e)}"
    cleanup_errors.append(error_msg)
    print(f"  ⚠ {error_msg}")

# 4. Delete Lambda Function
print("\n[4/7] Deleting Lambda function...")
try:
    lambda_client.delete_function(FunctionName=LAMBDA_FUNCTION)
    print(f"  ✓ Deleted Lambda: {LAMBDA_FUNCTION}")
except lambda_client.exceptions.ResourceNotFoundException:
    print(f"  ℹ Lambda {LAMBDA_FUNCTION} already deleted")
except Exception as e:
    error_msg = f"Lambda deletion: {str(e)}"
    cleanup_errors.append(error_msg)
    print(f"  ⚠ {error_msg}")

# 5. Delete Lambda IAM Role
print("\n[5/7] Deleting Lambda IAM role...")

# Delete inline policy
try:
    iam.delete_role_policy(
        RoleName=LAMBDA_ROLE,
        PolicyName='DynamoDBAccess'
    )
    print(f"  ✓ Deleted inline policy: DynamoDBAccess")
except iam.exceptions.NoSuchEntityException:
    print(f"  ℹ Policy DynamoDBAccess already deleted")
except Exception as e:
    print(f"  ⚠ Inline policy deletion: {str(e)}")

# Detach managed policy
try:
    iam.detach_role_policy(
        RoleName=LAMBDA_ROLE,
        PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
    )
    print(f"  ✓ Detached managed policy")
except iam.exceptions.NoSuchEntityException:
    print(f"  ℹ Policy already detached or role deleted")
except Exception as e:
    print(f"  ⚠ Managed policy detachment: {str(e)}")

# Delete role
try:
    iam.delete_role(RoleName=LAMBDA_ROLE)
    print(f"  ✓ Deleted role: {LAMBDA_ROLE}")
except iam.exceptions.NoSuchEntityException:
    print(f"  ℹ Role {LAMBDA_ROLE} already deleted")
except Exception as e:
    error_msg = f"IAM role deletion: {str(e)}"
    cleanup_errors.append(error_msg)
    print(f"  ⚠ {error_msg}")

# 6. Delete S3 Bucket
print("\n[6/7] Deleting S3 bucket...")
try:
    # Delete all objects first
    try:
        paginator = s3.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=S3_BUCKET)
        
        objects_deleted = 0
        for page in pages:
            if 'Contents' in page:
                objects = [{'Key': obj['Key']} for obj in page['Contents']]
                s3.delete_objects(
                    Bucket=S3_BUCKET,
                    Delete={'Objects': objects}
                )
                objects_deleted += len(objects)
        
        if objects_deleted > 0:
            print(f"  ✓ Deleted {objects_deleted} objects")
        else:
            print(f"  ℹ No objects to delete")
    except s3.exceptions.NoSuchBucket:
        print(f"  ℹ Bucket {S3_BUCKET} already deleted")
    
    # Delete bucket
    try:
        s3.delete_bucket(Bucket=S3_BUCKET)
        print(f"  ✓ Deleted bucket: {S3_BUCKET}")
    except s3.exceptions.NoSuchBucket:
        print(f"  ℹ Bucket {S3_BUCKET} already deleted")
    
except Exception as e:
    error_msg = f"S3 deletion: {str(e)}"
    cleanup_errors.append(error_msg)
    print(f"  ⚠ {error_msg}")

# 7. Delete DynamoDB Table
print("\n[7/7] Deleting DynamoDB table...")
try:
    dynamodb.delete_table(TableName=DYNAMODB_TABLE)
    print(f"  ✓ Deleted table: {DYNAMODB_TABLE}")
    print("  ⏳ Waiting for table deletion...")
    
    waiter = dynamodb.get_waiter('table_not_exists')
    waiter.wait(TableName=DYNAMODB_TABLE)
    print("  ✓ Table deletion complete")
    
except dynamodb.exceptions.ResourceNotFoundException:
    print(f"  ℹ Table {DYNAMODB_TABLE} already deleted")
except Exception as e:
    error_msg = f"DynamoDB deletion: {str(e)}"
    cleanup_errors.append(error_msg)
    print(f"  ⚠ {error_msg}")

# Delete SSM Parameters
try:
    print("\n[Extra] Deleting SSM parameters...")
    paginator = ssm.get_paginator('get_parameters_by_path')
    pages = paginator.paginate(Path=f"/{RESOURCE_PREFIX}", Recursive=True)
    
    param_names = []
    for page in pages:
        param_names.extend([p['Name'] for p in page['Parameters']])
    
    if param_names:
        for i in range(0, len(param_names), 10):
            batch = param_names[i:i+10]
            ssm.delete_parameters(Names=batch)
        print(f"  ✓ Deleted {len(param_names)} SSM parameters")
    else:
        print(f"  ℹ No SSM parameters to delete")
except Exception as e:
    print(f"  ⚠ SSM parameters deletion: {str(e)}")

# Summary
print("\n" + "="*70)
if cleanup_errors:
    print("CLEANUP COMPLETED WITH WARNINGS")
    print("="*70)
    print(f"\n⚠ {len(cleanup_errors)} warnings occurred:")
    for i, error in enumerate(cleanup_errors, 1):
        print(f"  {i}. {error}")
    print("\nNote: Some errors may require manual cleanup.")
else:
    print("CLEANUP COMPLETE - ALL RESOURCES DELETED")
    print("="*70)
    print("\n✓ All infrastructure has been successfully removed")

print("\nDeleted resources:")
print(f"  - AgentCore Gateway: {gateway_id}")
print(f"  - Gateway IAM Role: {GATEWAY_ROLE}")
print(f"  - Cognito User Pool: {user_pool_id}")
print(f"  - Cognito Resource Server: {RESOURCE_PREFIX}-resource-server")
print(f"  - Lambda Function: {LAMBDA_FUNCTION}")
print(f"  - Lambda IAM Role: {LAMBDA_ROLE}")
print(f"  - S3 Bucket: {S3_BUCKET}")
print(f"  - DynamoDB Table: {DYNAMODB_TABLE}")
print(f"  - SSM Parameters: /{RESOURCE_PREFIX}/*")
print("\n" + "="*70)


STARTING CLEANUP - DELETING ALL RESOURCES

[1/7] Deleting AgentCore Gateway...
  ✓ Deleted gateway target: D4NS4WNNAK
  ⏳ Waiting 10 seconds for target deletion to complete...
  ✓ All targets deleted
  ✓ Deleted Gateway: finance-analyzer-gateway-mnjaovrwqy

[2/7] Deleting Gateway IAM role...
  ✓ Deleted inline policy: LambdaInvokeAccess
  ✓ Deleted role: finance-analyzer-gateway-role

[3/7] Deleting Cognito resources...
  ✓ Deleted app client: c2v6c2jjvlq8ggm867rcq9eb2
  ✓ Deleted resource server: finance-analyzer-resource-server
  ✓ Deleted domain: finance-analyzer-domain
  ✓ Deleted user pool: us-east-1_CNm2wXuaJ

[4/7] Deleting Lambda function...
  ✓ Deleted Lambda: finance-analyzer-project-queries

[5/7] Deleting Lambda IAM role...
  ✓ Deleted inline policy: DynamoDBAccess
  ✓ Detached managed policy
  ✓ Deleted role: finance-analyzer-lambda-role

[6/7] Deleting S3 bucket...
  ✓ Deleted 1 objects
  ✓ Deleted bucket: finance-analyzer-data-961341522526

[7/7] Deleting DynamoDB table